# Defter 16 — İkinci hakem turu: modelleri aynı hedefte karşılaştırma, doğrudan fark testleri, blok duyarlılığı
**Detecting climate change in tree growth** · Prof. Dr. İlker Ercanlı

* **Q1 — Aynı hedef:** her modelin 1961–2019'da (yalnız 1902–1960 ile eğitilmiş) tahmin ettiği büyüme serisine gözlenen seriyle **aynı** uç olay analizi uygulanır;
  model L'si gözlenen L ile karşılaştırılır; T+S ile diğerleri arasındaki fark için blok bootstrap.
* **Q2 — Mekânsal dışarıda bırakma:** blok modellerinin test tahminleriyle aynı karşılaştırma (varsa).
* **Q3 — Tespit testi:** β farkları (eşleştirilmiş blok bootstrap); 5°/10°/20° blok duyarlılığı; blok ortak permütasyonu.
* **Q4 — Saha sayıları:** Şekil 3b ile 3c arasındaki farkın kaynağı.

**Girdi:** Defter 11–15 çıktıları; `zaman__*`, `tum__*`, `blok-*` modeller. Yeni eğitim yoktur. **Süre:** ≈ 40–70 dk (klasik yöntemler yeniden kalibre edilir).

In [ ]:
# --- Kurulum ---
import os, json, time, math, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ANA = '/content/drive/MyDrive/ITRDB_AI'
except ImportError:
    ANA = os.environ.get('ITRDB_AI', '.')
SONUC = os.path.join(ANA, 'SONUC_SON'); os.makedirs(SONUC, exist_ok=True)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Cihaz:', DEV, torch.cuda.get_device_name(0) if DEV == 'cuda' else '(GPU yok: Çalışma zamanı → T4 GPU seçin)')

AYAR = dict(batch=32768, max_epoch=25, sabir=3, lr=2e-3, lr_gizil=1e-2, J=8, gizli=128, emb=8,
            takson_dusur=0.10, su_onsel=0.5, tau_onsel=0.5, sv_onsel=0.5, desc_gurultu=0.2, wd=1e-4,
            rastgele_etki=False,   # False: popülasyon modeli (ana analiz); True: ağaç/alan rastgele etkili duyarlılık analizi
            r0_bas_cm=2.0, sifir_mm=0.005, hizli_deneme=False)

In [ ]:
# --- Veri ---
d = np.load(os.path.join(ANA, 'TENSOR', 'veri_v2.npz'))
soz = json.load(open(os.path.join(ANA, 'TENSOR', 'veri_v2_sozluk.json')))
YIL0 = soz['yil0']; S = len(soz['site_code']); NG = len(soz['genus']); NS = len(soz['species'])
NSER = len(d['series_site']); NTREE = int(d['series_tree'].max()) + 1
t = lambda x, dt=None: torch.as_tensor(np.asarray(x), dtype=dt).to(DEV)

# Halkalar: 1901 halkaları çıkar (önceki yılın iklimi yok)
k = d['ring_year'] > YIL0
R_ser = t(d['ring_series'][k], torch.long); R_yil = t(d['ring_year'][k], torch.long)
R_w = t(d['ring_rw'][k], torch.float32); R_rp = t(d['ring_rprev'][k], torch.float32) / 10   # cm
ser_site = t(d['series_site'], torch.long); ser_tree = t(d['series_tree'], torch.long)
ser_zaman = t(d['series_zaman'], torch.bool)
site_genus = t(d['site_genus'], torch.long); site_species = t(d['site_species'], torch.long)
NR = len(R_w)

# Aylık iklim özellikleri [alan, ay(124*12), kanal]: T, log(1+P), M, VPD, gE
clim = d['clim']; soil = d['soil']; gE = d['gE']
NY = clim.shape[1]
ozel = np.stack([clim[..., 0], np.log1p(np.maximum(clim[..., 1], 0)), soil, clim[..., 3],
                 np.broadcast_to(gE[:, None, :], soil.shape)], -1).reshape(S, NY * 12, 5).astype(np.float32)
OZ = t(ozel); del ozel, clim
oz_ort = OZ.reshape(-1, 5).mean(0); oz_ss = OZ.reshape(-1, 5).std(0)
desc = d['desc']; DESC = t((desc - desc.mean(0)) / desc.std(0), torch.float32)
# Yetişme ortamı: 11 sürekli değişken (standartlaştırılmış) + biyom, ekolojik bölge, Köppen sınıfları (son indeks = bilinmeyen)
env = d['env']; ENV = t((env - env.mean(0)) / env.std(0), torch.float32)
site_biyom = t(d['site_biyom'], torch.long); site_eko = t(d['site_eko'], torch.long); site_kg = t(d['site_kg'], torch.long)
NB, NE, NK = soz['n_biyom'], soz['n_eko'], soz['n_kg']
KAT_GOR = {}   # bölünmeye göre eğitimde görülen sınıflar (görülmeyenler 'bilinmeyen' gömmesine gider)
ORTAM_BOYUT = ENV.shape[1] + 3 * 4

class OrtamGirdi(nn.Module):
    """Alan başına yetişme ortamı vektörü: sürekli değişkenler + biyom/ekolojik bölge/Köppen gömmeleri (her biri 4 boyut)"""
    def __init__(self):
        super().__init__()
        self.b = nn.Embedding(NB + 1, 4); self.e = nn.Embedding(NE + 1, 4); self.k = nn.Embedding(NK + 1, 4)
    def forward(self, G):
        b, e, k = G['b'], G['e'], G['k']
        if KAT_GOR:
            b = torch.where(KAT_GOR['b'][b], b, torch.full_like(b, NB)); e = torch.where(KAT_GOR['e'][e], e, torch.full_like(e, NE))
            k = torch.where(KAT_GOR['k'][k], k, torch.full_like(k, NK))
        if self.training:      # %10 rastgele maskeleme: 'bilinmeyen' gömmesi de öğrenilsin
            m = torch.rand(len(b), device=DEV) < 0.10
            b = torch.where(m, torch.full_like(b, NB), b); e = torch.where(m, torch.full_like(e, NE), e); k = torch.where(m, torch.full_like(k, NK), k)
        x = G['env'] + (AYAR['desc_gurultu'] * torch.randn_like(G['env']) if self.training else 0)
        return torch.cat([x, self.b(b), self.e(e), self.k(k)], 1)
print(f'{NR:,} halka, {NSER:,} seri, {NTREE:,} ağaç, {S} alan — veri {"GPU" if DEV=="cuda" else "CPU"} belleğinde')

# Yerel uyum için alan nem normali: 1901–1960 aylık toprak nemi ortalaması (zamansal testte de yalnız eğitim dönemi)
MBAR = OZ[:, :60 * 12, 2].mean(1); MREF = MBAR.mean()
def alan_girdi(s):
    """Alan düzeyi girdiler (SHAP ve ihlal analizinde gruplar halinde değiştirilebilsin diye sözlük)"""
    return dict(g=site_genus[s], sp=site_species[s], desc=DESC[s], env=ENV[s], b=site_biyom[s], e=site_eko[s], k=site_kg[s], mbar=MBAR[s])

def girdiler(idx):
    """Halka düzeyi girdiler: 24 aylık iklim penceresi X, birikmiş yarıçap rp, seri ve alan girdileri"""
    X, s = pencere(idx); G = alan_girdi(s); G.update(X=X, rp=R_rp[idx], ser=R_ser[idx]); return G

def pencere(idx):
    """Halka idx için 24 aylık pencere (önceki yıl Ocak – bu yıl Aralık): [B, 24, 5]"""
    s = ser_site[R_ser[idx]]
    bas = (R_yil[idx] - 1 - YIL0) * 12
    ay = bas[:, None] + torch.arange(24, device=DEV)[None, :]
    return OZ[s[:, None], ay], s

In [ ]:
# --- Test bölünmeleri ---
site_bolge = np.array(soz['site_bolge']); site_kat = d['site_blok_kat']; site_takson = d['site_takson_test']
site_ic = d['site_ic_dogrulama']

def bolunme(ad):
    """Dönüş: (eğitim, doğrulama, test) halka indeksleri ve test alanlarında takson gizlensin mi"""
    site_r = ser_site[R_ser].cpu().numpy(); yil = R_yil.cpu().numpy(); zuy = ser_zaman[R_ser].cpu().numpy()
    gizle = False
    if ad == 'zaman':
        test = zuy & (yil > 1960); havuz = yil <= 1960
    elif ad.startswith('bolge:'):
        tb = np.isin(site_r, np.where(site_bolge == ad.split(':')[1])[0]); test = tb; havuz = ~tb
    elif ad.startswith('blok:'):
        tb = np.isin(site_r, np.where(site_kat == int(ad.split(':')[1]))[0]); test = tb; havuz = ~tb
    elif ad == 'takson':
        tb = np.isin(site_r, np.where(site_takson)[0]); test = tb; havuz = ~tb; gizle = True
    elif ad.startswith('iklim:'):
        # İklim dışa-genelleme: en sıcak (1901–1960 ort. sıcaklık) ya da en kurak (P/PET) %10 alan yalnız testte
        tur = ad.split(':')[1]
        if tur == 'kurak2':          # %10–20 kurak bant test; eğitim yalnız en kurak %20 dışı alanlarla
            x = -desc[:, 5]; q8, q9 = np.quantile(x, [0.8, 0.9])
            test = np.isin(site_r, np.where((x >= q8) & (x < q9))[0]); havuz = np.isin(site_r, np.where(x < q8)[0])
        else:
            x = {'sicak': desc[:, 2], 'kurak': -desc[:, 5], 'nemli': desc[:, 5]}[tur]
            tb = np.isin(site_r, np.where(x >= np.quantile(x, 0.9))[0]); test = tb; havuz = ~tb
    elif ad == 'tum':            # projeksiyon için son modeller: tüm alanlar ve yıllar; test = iç doğrulama alanları
        havuz = np.ones(len(site_r), bool); test = site_ic[site_r].copy()
    else:
        raise ValueError(ad)
    ic = site_ic[site_r]
    egit = np.where(havuz & ~ic)[0]; dog = np.where(havuz & ic)[0]; test = np.where(test)[0]
    return t(egit, torch.long), t(dog, torch.long), t(test, torch.long), gizle

In [ ]:
# --- Modeller ---
class AlanAgi(nn.Module):
    """z_s -> parametreler. Cins/tür gömmeleri + standartlaştırılmış alan tanımlayıcıları."""
    def __init__(self, n_cikis, emb, gizli, sutun=None, ortam=False):
        super().__init__()
        self.ortam = OrtamGirdi() if ortam else None
        self.sutun = list(range(DESC.shape[1])) if sutun is None else sutun   # saf teori: yalnız [enlem, yükselti]
        self.ge = nn.Embedding(NG + 1, emb); self.se = nn.Embedding(NS + 1, emb)   # son indeks = "bilinmeyen takson"
        self.mlp = nn.Sequential(nn.Linear(2 * emb + len(self.sutun) + (ORTAM_BOYUT if ortam else 0), gizli), nn.GELU(),
                                 nn.Linear(gizli, gizli), nn.GELU(), nn.Linear(gizli, n_cikis))
    def forward(self, G, gizle_maske=None):
        g, sp = G['g'], G['sp']
        if gizle_maske is not None:
            g = torch.where(gizle_maske, torch.full_like(g, NG), g); sp = torch.where(gizle_maske, torch.full_like(sp, NS), sp)
        d = G['desc'][:, self.sutun]
        d = d + (AYAR['desc_gurultu'] * torch.randn_like(d) if self.training else 0)
        parcalar = [self.ge(g), self.se(sp), d] + ([self.ortam(G)] if self.ortam is not None else [])
        return self.mlp(torch.cat(parcalar, 1))

def monoton(x, a, c, s):
    """Denklem (4): negatif olmayan lojistik karışımı, 0–1 arası, azalmayan. x:[B,24], a,c,s:[B,J]"""
    w = F.softplus(a) + 1e-4
    z = torch.sigmoid((x[..., None] - c[:, None, :]) / s[:, None, :])
    return (z * w[:, None, :]).sum(-1) / w.sum(-1, keepdim=True)

def alan_gor_bul(seri_gor):
    m = torch.zeros(S, dtype=torch.bool, device=DEV); m[ser_site[seri_gor]] = True
    return m

class Gizil(nn.Module):
    """Seri başına gizil başlangıç yarıçapı r0 = softplus(ρ) ve ağaç etkisi u; hiyerarşik önsel cezası"""
    def __init__(self, r0_bas):
        super().__init__()
        rho0 = math.log(math.expm1(r0_bas))
        self.rho = nn.Parameter(torch.full((NSER,), rho0, device=DEV))
        self.mu_g = nn.Parameter(torch.full((NG + 1,), rho0, device=DEV))
        self.u = nn.Parameter(torch.zeros(NTREE, device=DEV))          # ağaç etkisi
        self.v = nn.Parameter(torch.zeros(S, device=DEV))              # alan etkisi (toprak, sıklık vb. kaydedilmemiş farklar)
        # Önsel ölçekleri sabit (zayıf bilgili) hiperparametreler: MAP ile öğrenilen varyans bileşenleri sıfıra çöker.
        # Yeni ağaç/seri için öngörüde kullanılan yayılım ise eğitilmiş ağaçların GÖRGÜL dağılımından alınır.
        self.su_onsel, self.tau_onsel = AYAR['su_onsel'], AYAR['tau_onsel']
        self.register_buffer('su_gorgul', torch.tensor(AYAR['su_onsel'], device=DEV))
        self.register_buffer('tau_gorgul', torch.tensor(AYAR['tau_onsel'], device=DEV))
        self.sv_onsel = AYAR['sv_onsel']
        self.register_buffer('sv_gorgul', torch.tensor(AYAR['sv_onsel'], device=DEV))
        self.register_buffer('ort_u', torch.tensor(0., device=DEV)); self.register_buffer('ort_v', torch.tensor(0., device=DEV))
        self.register_buffer('ort_d', torch.tensor(0., device=DEV))
    def su(self): return self.su_gorgul
    def tau(self): return self.tau_gorgul
    def gorgul_guncelle(self, seri_gor, agac_gor):
        with torch.no_grad():
            g = site_genus[ser_site]
            # Görgül dağılım: ortalama da alınır (ağın sabit terimi ile rastgele etkilerin ortalaması birbirine kayabilir)
            u, d, v = self.u[agac_gor], (self.rho - self.mu_g[g])[seri_gor], self.v[alan_gor_bul(seri_gor)]
            self.su_gorgul.copy_(u.std().clamp_min(0.02)); self.ort_u.copy_(u.mean())
            self.tau_gorgul.copy_(d.std().clamp_min(0.02)); self.ort_d.copy_(d.mean())
            self.sv_gorgul.copy_(v.std().clamp_min(0.02)); self.ort_v.copy_(v.mean())
    def ceza(self, seri_gor, agac_gor):
        """Hiyerarşik önsel; YALNIZ eğitimde verisi olan seri ve ağaçlar üzerinden"""
        g = site_genus[ser_site]
        d = (self.rho - self.mu_g[g])[seri_gor]; u = self.u[agac_gor]; v = self.v[alan_gor_bul(seri_gor)]
        return ((d ** 2).sum() / (2 * self.tau_onsel ** 2) + (u ** 2).sum() / (2 * self.su_onsel ** 2)
                + (v ** 2).sum() / (2 * self.sv_onsel ** 2))
    def gizil_deger(self, ser, ornek=None):
        """Halka başına ρ ve u. ornek=(seri_gor, agac_gor, gizle): görülmemişler için önselden rastgele çekim"""
        rho, u, v = self.rho[ser], self.u[ser_tree[ser]], self.v[ser_site[ser]]
        if ornek is not None:
            seri_gor, agac_gor, gizle, mu_glob = ornek
            g = site_genus[ser_site[ser]]
            mu = torch.where(gizle, mu_glob, self.mu_g[g]) if gizle is not None else self.mu_g[g]
            rho = torch.where(seri_gor[ser], rho, mu + self.ort_d + self.tau() * torch.randn_like(rho))
            u = torch.where(agac_gor[ser_tree[ser]], u, self.ort_u + self.su() * torch.randn_like(u))
            v = torch.where(alan_gor_bul(seri_gor)[ser_site[ser]], v, self.ort_v + self.sv_gorgul * torch.randn_like(v))
        return rho, u + v

class TeoriModeli(nn.Module):
    def __init__(self, J=8, emb=8, gizli=128, ablasyon=None, r0_bas=2.0, m_bas=2/3, vpd=False, m_sabit=None, saf=False, ortam=False, uyum=False, genis=False, tur=False):
        super().__init__()
        self.J, self.abl = J, ablasyon
        self.uyum = uyum
        self.mgen = 0.3 if genis else 0.1   # nem eşiği merkezlerinin kayma aralığı
        self.m_logit = math.log(m_bas / (1 - m_bas)); self.vpd = vpd; self.m_sabit = m_sabit
        n = 3 + 6 * J + 24 + 1 + 1          # +1: başlangıç yarıçapı r0 (alan ve takson düzeyinde, ağdan)
        if ablasyon == 'abl_min': n += 3
        if vpd: n += 3 * J                 # VPD yanıtı (üçüncü sınırlayıcı faktör)
        if uyum: n += 1                    # yerel uyum katsayısı λ
        self.Dbaz = torch.linspace(0.2, 3.0, J, device=DEV)
        self.ag = AlanAgi(n, emb, gizli, sutun=[] if tur else ([0, 1] if saf else None), ortam=ortam)   # saf: iklim normalleri parametre ağına girmez
        if ablasyon == 'abl_mono':      # monoton yanıt yerine kısıtsız küçük ağ (girdi: x ve alan parametreleri)
            self.serbT = nn.Sequential(nn.Linear(1 + 3 * J, 32), nn.GELU(), nn.Linear(32, 1))
            self.serbM = nn.Sequential(nn.Linear(1 + 3 * J, 32), nn.GELU(), nn.Linear(32, 1))
        self.gz = Gizil(r0_bas)
        self.Tbaz = torch.linspace(-5, 25, J, device=DEV); self.Mbaz = torch.linspace(0.02, 0.6, J, device=DEV)

    def parametreler(self, G, gizle=None):
        if torch.is_tensor(G): G = alan_girdi(G)
        o = self.ag(G, gizle); J = self.J
        p = {}
        p['eta'] = torch.exp(o[:, 0] + math.log(0.7))
        p['Bmax'] = torch.exp(o[:, 1] + math.log(1e4))                         # cm² (başlangıç: r ≈ 56 cm)
        p['m'] = torch.sigmoid(o[:, 2] + self.m_logit)                            # başlangıç varsayılan 2/3
        if self.m_sabit is not None: p['m'] = torch.full_like(p['m'], self.m_sabit)   # profil olabilirlik: m sabit
        if getattr(self, 'm_cins', None) is not None:   # sentetik veri üretimi: m cinse göre dışarıdan; η aynı ortalama büyümeyi korur
            m_yeni = self.m_cins[G['g']]; p['eta'] = p['eta'] * 300.0 ** (p['m'] - m_yeni); p['m'] = m_yeni
        i = 3
        p['aT'], p['cT'], p['sT'] = o[:, i:i+J], self.Tbaz + 5 * torch.tanh(o[:, i+J:i+2*J]), 0.5 + 2 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        p['aM'], p['cM'], p['sM'] = o[:, i:i+J], self.Mbaz + self.mgen * torch.tanh(o[:, i+J:i+2*J]), 0.005 + 0.03 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        p['omega'] = torch.softmax(o[:, i:i+24], 1); i += 24
        if self.vpd:
            p['aD'], p['cD'], p['sD'] = o[:, i:i+J], self.Dbaz + 0.3 * torch.tanh(o[:, i+J:i+2*J]), 0.05 + 0.3 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        p['log_sigma'] = o[:, i] - 0.3; i += 1
        p['r0'] = F.softplus(o[:, i] + math.log(math.expm1(AYAR['r0_bas_cm']))); i += 1   # cm
        if self.abl == 'abl_min': p['karisim'] = torch.softmax(o[:, i:i+3], 1)
        if self.uyum: p['lam'] = torch.sigmoid(o[:, -1])          # 0 = mutlak tepki, 1 = tamamen yerel uyumlu
        return p

    def forward(self, idx, gizle=None, ornek=None, G=None):
        G = girdiler(idx) if G is None else G
        X = G['X']; p = self.parametreler(G, gizle); G0 = G
        ser = G['ser']
        rho_i, u_i = self.gz.gizil_deger(ser, ornek)
        if not AYAR['rastgele_etki']: u_i = torch.zeros_like(u_i)
        r0 = 0.05 if self.abl == 'abl_r0' else (F.softplus(rho_i) if AYAR['rastgele_etki'] else p['r0'])
        r = r0 + G['rp']; B = math.pi * r ** 2                                   # cm²
        # Denklem (2): büyüme potansiyeli
        if self.abl == 'abl_G':
            G = p['eta'] * 30.0
        elif self.abl == 'abl_us':        # kuvvet fonksiyonlu potansiyel: G = η B^m (B_max yok)
            G = p['eta'] * B.clamp_min(1e-6) ** p['m']
        else:
            x = (B / p['Bmax']).clamp_min(1e-8) ** (1 - p['m'])
            G = p['eta'] * B.clamp_min(1e-6) ** p['m'] * F.softplus(20 * (1 - x)) / 20
        # Denklem (3)–(4): sınırlayıcı faktör
        T, M, gEk = X[..., 0], X[..., 2], X[..., 4]
        if self.uyum: M = M - p['lam'][:, None] * (G0['mbar'][:, None] - MREF)   # eşik alanın nem normaline göre kayar
        if self.abl == 'abl_mono':
            pT = torch.cat([p['aT'], p['cT'], p['sT']], 1); pM = torch.cat([p['aM'], p['cM'], p['sM']], 1)
            gT = torch.sigmoid(self.serbT(torch.cat([T[..., None] / 10, pT[:, None, :].expand(-1, 24, -1)], -1))[..., 0])
            gM = torch.sigmoid(self.serbM(torch.cat([M[..., None] * 5, pM[:, None, :].expand(-1, 24, -1)], -1))[..., 0])
        else:
            gT = monoton(T, p['aT'], p['cT'], p['sT']); gM = monoton(M, p['aM'], p['cM'], p['sM'])
        if self.abl == 'abl_min':
            k = p['karisim'][:, None, :]
            sinir = k[..., 0] * gT + k[..., 1] * gM + k[..., 2] * gT * gM
        else:
            sinir = torch.minimum(gT, gM)
        if self.vpd:     # Liebig: min(g_T, g_M, g_D); g_D = 1 − monoton(VPD) → atmosfer kurudukça büyüme azalır
            gD = 1 - monoton(X[..., 3], p['aD'], p['cD'], p['sD'])
            sinir = torch.minimum(sinir, gD)
        C = (p['omega'] * gEk * sinir).sum(1).clamp_min(1e-6)
        BAI = G * C * torch.exp(u_i)
        w_hat = (torch.sqrt((B + BAI) / math.pi) - r) * 10                         # mm
        mu = torch.log(w_hat.clamp_min(1e-4))
        return mu, p['log_sigma'], dict(gT=gT, gM=gM, C=C, G=G, lam=p.get('lam'))

class SerbestAg(nn.Module):
    """Kısıtsız karşılaştırma: 24 aylık iklim -> zamansal evrişim; + log yarıçap + alan gömmesi -> log w"""
    def __init__(self, emb=8, gizli=40, r0_bas=2.0, ortam=False):
        super().__init__()
        self.ortam = OrtamGirdi() if ortam else None
        self.conv = nn.Sequential(nn.Conv1d(5, 32, 3), nn.GELU(), nn.Conv1d(32, 32, 3), nn.GELU())
        self.ge = nn.Embedding(NG + 1, emb); self.se = nn.Embedding(NS + 1, emb)
        self.mlp = nn.Sequential(nn.Linear(32 * 20 + 2 * emb + DESC.shape[1] + 1 + (ORTAM_BOYUT if ortam else 0), gizli), nn.GELU(),
                                 nn.Linear(gizli, gizli), nn.GELU(), nn.Linear(gizli, 2))
        self.gz = Gizil(r0_bas)
    def forward(self, idx, gizle=None, ornek=None, G=None):
        G = girdiler(idx) if G is None else G
        X = (G['X'] - oz_ort) / oz_ss
        h = self.conv(X.transpose(1, 2)).flatten(1)
        g, sp = G['g'], G['sp']
        if gizle is not None:
            g = torch.where(gizle, torch.full_like(g, NG), g); sp = torch.where(gizle, torch.full_like(sp, NS), sp)
        lr = torch.log1p(G['rp'])[:, None]
        d = G['desc'] + (AYAR['desc_gurultu'] * torch.randn_like(G['desc']) if self.training else 0)
        parcalar = [h, self.ge(g), self.se(sp), d, lr] + ([self.ortam(G)] if self.ortam is not None else [])
        o = self.mlp(torch.cat(parcalar, 1))
        _, u_i = self.gz.gizil_deger(G['ser'], ornek)
        if not AYAR['rastgele_etki']: u_i = torch.zeros_like(u_i)
        return o[:, 0] - 0.3 + u_i, o[:, 1] - 0.3, {}

def yap(ad):
    """Model adı: taban ad + isteğe bağlı '_ortam' eki (yetişme ortamı girdileri)"""
    ortam = ad.endswith('_ortam'); ad = ad[:-6] if ortam else ad
    T = lambda **k: TeoriModeli(AYAR['J'], AYAR['emb'], AYAR['gizli'], k.pop('abl', None), AYAR['r0_bas_cm'], ortam=ortam, **k).to(DEV)
    if ad == 'serbest': return SerbestAg(AYAR['emb'], r0_bas=AYAR['r0_bas_cm'], ortam=ortam).to(DEV)
    if ad == 'teori': return T()
    if ad == 'teori_vpd': return T(vpd=True)
    if ad == 'teori_uyum': return T(uyum=True)
    if ad == 'teori_genis': return T(genis=True)
    if ad == 'teori_tur': return T(tur=True)
    if ad == 'teori_saf': return T(vpd=True, saf=True)
    if ad.startswith('teori_mf'):       # profil olabilirlik: teori_mf050 -> m = 0.50 sabit
        return T(m_sabit=int(ad[8:]) / 1000 if len(ad) == 12 else int(ad[8:]) / 100)
    if ad.startswith('teori_m'): return T(m_bas=int(ad[7:]) / 100)
    if ad.startswith('abl_'): return T(abl=ad)
    raise ValueError(ad)

def ag_param_sayisi(m):
    return sum(p.numel() for n, p in m.named_parameters() if not n.startswith('gz.'))

for ad in ['teori_ortam', 'teori_tur']:
    print(f'{ad:16s} eğitilebilir ağ parametresi: {ag_param_sayisi(yap(ad)):,}')

In [ ]:
# --- Eğitim ---
LOG_Y_SIFIR = math.log(AYAR['sifir_mm'])
def hedef(idx):
    w = R_w[idx]
    return torch.where(w > 0, torch.log(w.clamp_min(1e-6)), torch.full_like(w, LOG_Y_SIFIR))

LOG_ESIK = math.log(0.01)            # ölçüm çözünürlüğü: sıfır halka = "0,01 mm'den küçük" (sansürlü gözlem)
def nll(mu, ls, y):
    sifir = y <= LOG_Y_SIFIR + 1e-6
    tam = 0.5 * ((y - mu) / torch.exp(ls)) ** 2 + ls + 0.9189385
    sans = -torch.special.log_ndtr((LOG_ESIK - mu) / torch.exp(ls))
    return torch.where(sifir, sans, tam).mean()

@torch.no_grad()
def tahmin(model, idx, seri_gor, agac_gor, gizle_alan=False, K=8, bs=131072):
    """Görülmüş ağaçlar: koşullu öngörü. Görülmemiş ağaçlar (yeni alan/tür): ρ ve u önselden K kez çekilir;
    öngörü = karışımın ortalaması, varyans = ort(σ²) + var(μ)  (log ölçeğinde)"""
    model.eval(); MU, SG = [], []
    g_gor = torch.unique(site_genus[ser_site[seri_gor.nonzero().squeeze(1)]])
    mu_glob = model.gz.mu_g[g_gor].mean()
    with torch.no_grad():
        for i in range(0, len(idx), bs):
            b = idx[i:i+bs]
            gz = torch.ones(len(b), dtype=torch.bool, device=DEV) if gizle_alan else None
            yeni = ~agac_gor[ser_tree[R_ser[b]]] | ~seri_gor[R_ser[b]]
            if not AYAR['rastgele_etki']: yeni = torch.zeros_like(yeni)
            if not yeni.any():
                mu, ls, _ = model(b, gz); MU.append(mu); SG.append(torch.exp(ls)); continue
            ms, vs = [], []
            for k in range(K):
                mu, ls, _ = model(b, gz, ornek=(seri_gor, agac_gor, gz, mu_glob)); ms.append(mu); vs.append(torch.exp(2 * ls))
            ms, vs = torch.stack(ms), torch.stack(vs)
            MU.append(ms.mean(0)); SG.append(torch.sqrt(vs.mean(0) + ms.var(0, unbiased=False)))
    model.train()
    return torch.cat(MU), torch.cat(SG)

def kat_gor_ayarla(egt):
    """Eğitim halkalarının alanlarında görülen biyom / ekolojik bölge / Köppen sınıfları"""
    sg = torch.unique(ser_site[R_ser[egt]]); KAT_GOR.clear()
    for anahtar, dizi, n in [('b', site_biyom, NB), ('e', site_eko, NE), ('k', site_kg, NK)]:
        m = torch.zeros(n + 1, dtype=torch.bool, device=DEV); m[dizi[sg]] = True; KAT_GOR[anahtar] = m

def egit(bolunme_ad, model_ad, tohum):
    is_ad = f'{bolunme_ad.replace(":", "-")}__{model_ad}__s{tohum}'
    klasor = os.path.join(SONUC, is_ad); os.makedirs(klasor, exist_ok=True)
    if os.path.exists(os.path.join(klasor, 'bitti.json')):
        print('atlandı (bitmiş):', is_ad); return
    torch.manual_seed(tohum); np.random.seed(tohum)
    egt, dog, test, gizle = bolunme(bolunme_ad)
    kat_gor_ayarla(egt)
    if AYAR['hizli_deneme']:
        egt, dog, test = egt[torch.randperm(len(egt), device=DEV)[:200000]], dog[:20000], test[:20000]
    model = yap(model_ad)
    gizil = [p for n, p in model.named_parameters() if n.startswith('gz.')]
    ag = [p for n, p in model.named_parameters() if not n.startswith('gz.')]
    opt = torch.optim.AdamW([{'params': ag, 'lr': AYAR['lr'], 'weight_decay': AYAR['wd']},
                             {'params': gizil, 'lr': AYAR['lr_gizil'] if AYAR['rastgele_etki'] else 0.0, 'weight_decay': 0.0}])
    kayit = os.path.join(klasor, 'son.pt'); bas_ep, en_iyi, sabir, gecmis = 0, float('inf'), 0, []
    if os.path.exists(kayit):
        ck = torch.load(kayit, map_location=DEV)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
        bas_ep, en_iyi, sabir, gecmis = ck['epoch'] + 1, ck['en_iyi'], ck['sabir'], ck['gecmis']
        print(f'{is_ad}: {bas_ep}. epoch\'tan devam')
    n_egt = len(egt)
    agac_gor = torch.zeros(NTREE, dtype=torch.bool, device=DEV); agac_gor[ser_tree[R_ser[egt]]] = True
    seri_gor = torch.zeros(NSER, dtype=torch.bool, device=DEV); seri_gor[R_ser[egt]] = True
    for ep in range(bas_ep, AYAR['max_epoch']):
        t0 = time.time(); kar = egt[torch.randperm(n_egt, device=DEV)]; top = 0.
        for i in range(0, n_egt, AYAR['batch']):
            b = kar[i:i+AYAR['batch']]
            gz = (torch.rand(len(b), device=DEV) < AYAR['takson_dusur'])
            mu, ls, _ = model(b, gz)
            kayip_v = nll(mu, ls, hedef(b))
            kayip = kayip_v + (model.gz.ceza(seri_gor, agac_gor) / n_egt if AYAR['rastgele_etki'] else 0.)
            opt.zero_grad(set_to_none=True); kayip.backward()
            torch.nn.utils.clip_grad_norm_(ag, 5.0); opt.step(); top += kayip_v.item() * len(b)
        model.gz.gorgul_guncelle(seri_gor, agac_gor)
        mu, sg = tahmin(model, dog, seri_gor, agac_gor)
        yd = hedef(dog); v = nll(mu, torch.log(sg), yd).item()
        nz = yd > LOG_Y_SIFIR + 1e-6; v_rmse = torch.sqrt(((mu - yd)[nz] ** 2).mean()).item(); v_sg = torch.sqrt((sg ** 2).mean()).item()
        gecmis.append(dict(epoch=ep, egitim=top / n_egt, dogrulama=v, sn=time.time() - t0))
        ek = (f'  σ_alan {model.gz.sv_gorgul.item():.2f}  σ_ağaç {model.gz.su().item():.2f}' if AYAR['rastgele_etki'] else '')
        print(f'{is_ad} ep {ep:2d}  eğitim {top/n_egt:.4f}  doğrulama {v:.4f} (RMSE {v_rmse:.3f}, öngörü σ {v_sg:.3f}){ek}  ({time.time()-t0:.0f} sn)')
        if v < en_iyi - 1e-4:
            en_iyi, sabir = v, 0; torch.save(model.state_dict(), os.path.join(klasor, 'en_iyi.pt'))
        else:
            sabir += 1
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), epoch=ep, en_iyi=en_iyi, sabir=sabir, gecmis=gecmis), kayit)
        if sabir >= AYAR['sabir']: break
    # Test tahminleri (en iyi model)
    model.load_state_dict(torch.load(os.path.join(klasor, 'en_iyi.pt'), map_location=DEV))
    mu, sg = tahmin(model, test, seri_gor, agac_gor, gizle_alan=gizle)
    y = hedef(test)
    np.savez_compressed(os.path.join(klasor, 'test_tahmin.npz'), idx=test.cpu().numpy().astype(np.int32),
                        mu=mu.cpu().numpy().astype(np.float16), sigma=sg.cpu().numpy().astype(np.float16))
    nz = y > LOG_Y_SIFIR + 1e-6         # nokta ölçütleri sıfır olmayan halkalarda; NLL sansürlü olanları da içerir
    ozet = dict(is_ad=is_ad, n_egitim=n_egt, n_test=len(test), rmse_log=float(torch.sqrt(((mu - y)[nz] ** 2).mean())),
                bias_log=float((mu - y)[nz].mean()), nll=float(nll(mu, torch.log(sg), y)),
                kapsama90=float(((y - mu).abs() < 1.645 * sg)[nz].float().mean()), en_iyi_dogrulama=en_iyi,
                ag_param=ag_param_sayisi(model), gecmis=gecmis, ayar=AYAR)
    if not model_ad.startswith('serbest'):           # alan parametreleri (m, eşikler, ω) — şekiller için
        with torch.no_grad():
            p = model.parametreler(torch.arange(S, device=DEV), torch.full((S,), gizle, device=DEV) & t(site_takson, torch.bool))
        np.savez_compressed(os.path.join(klasor, 'alan_param.npz'), **{k: v.cpu().numpy() for k, v in p.items()},
                            r0_cm=F.softplus(model.gz.rho).detach().cpu().numpy(), u=model.gz.u.detach().cpu().numpy())
    json.dump(ozet, open(os.path.join(klasor, 'bitti.json'), 'w'), indent=1)
    os.remove(kayit)
    print(f'BİTTİ {is_ad}: test RMSE(log) {ozet["rmse_log"]:.3f}, kapsama %90 → {ozet["kapsama90"]:.3f}')

In [ ]:
# --- Ortak araçlar: topluluk tahmini, ölçütler, bootstrap ---
import glob, pandas as pd
from scipy.stats import norm
ANALIZ = os.path.join(ANA, 'ANALIZ_SON'); os.makedirs(ANALIZ, exist_ok=True)
Yn = hedef(torch.arange(NR, device=DEV)).cpu().numpy(); NZn = (R_w > 0).cpu().numpy()
SITE_n = ser_site[R_ser].cpu().numpy(); YIL_n = R_yil.cpu().numpy()
def topluluk(b, m):
    MU, V, idx = [], [], None
    for kl in sorted(glob.glob(os.path.join(SONUC, f'{b}__{m}__s*'))):
        if not os.path.exists(os.path.join(kl, 'bitti.json')): continue
        z = np.load(os.path.join(kl, 'test_tahmin.npz'))
        if idx is None: idx = z['idx']
        MU.append(z['mu'].astype(np.float32)); V.append(z['sigma'].astype(np.float32) ** 2)
    if not MU: return None
    MU, V = np.stack(MU), np.stack(V)
    return idx, MU.mean(0), np.sqrt(V.mean(0) + MU.var(0)), len(MU)
def olcutler(y, mu, sg, nz):
    r = (y - mu)[nz]; z = (y - mu) / sg
    nllv = np.where(nz, 0.5 * z**2 + np.log(sg) + 0.9189385, -norm.logcdf((math.log(0.01) - mu) / sg))
    crps = sg * (z * (2 * norm.cdf(z) - 1) + 2 * norm.pdf(z) - 1 / math.sqrt(math.pi))
    return dict(RMSE=float(np.sqrt((r**2).mean())), bias=float(r.mean()), NLL=float(nllv.mean()),
                CRPS=float(crps[nz].mean()), kapsama90=float((np.abs(z) < 1.645)[nz].mean()))
rng = np.random.default_rng(2026)
def boot_fark(y, ma, mb, nz, grup, B=1000):
    """RMSE(a) − RMSE(b) ve grup (alan ya da yıl) bootstrap aralıkları"""
    g = grup[nz]; ea, eb = (y - ma)[nz] ** 2, (y - mb)[nz] ** 2
    u, inv = np.unique(g, return_inverse=True)
    Sa, Sb, n = np.bincount(inv, ea, len(u)), np.bincount(inv, eb, len(u)), np.bincount(inv, None, len(u))
    W = rng.multinomial(len(u), np.full(len(u), 1 / len(u)), size=B).astype(float)
    d = np.sqrt(W @ Sa / (W @ n)) - np.sqrt(W @ Sb / (W @ n))
    return float(np.sqrt(Sa.sum() / n.sum()) - np.sqrt(Sb.sum() / n.sum())), np.quantile(d, [0.025, 0.975]), np.quantile(d, [0.05, 0.95])
def alan_agirlikli_rmse(y, mu, nz, s):
    u, inv = np.unique(s[nz], return_inverse=True)
    return float(np.sqrt(np.bincount(inv, (y - mu)[nz] ** 2) / np.bincount(inv)).mean())

In [ ]:
# --- Senaryo ve özellik altyapısı (Defter 5–7 ile aynı) ---
from scipy.stats import spearmanr
import gc
CL = d['clim']; LAT = d['site_lat']
def gun_uzunlugu(lat):
    gun = np.array([15, 46, 74, 105, 135, 166, 196, 227, 258, 288, 319, 349])
    dek = 23.45 * np.sin(np.deg2rad(360 * (284 + gun) / 365))
    x = -np.tan(np.deg2rad(lat))[:, None] * np.tan(np.deg2rad(dek))[None, :]
    return 24 / np.pi * np.arccos(np.clip(x, -1, 1))
def isi_indeksi(T):
    S_ = T.shape[0]; Tclim = np.nanmean(T.reshape(S_, -1, 12), axis=1)
    return np.sum(np.where(Tclim > 0, (np.maximum(Tclim, 0) / 5) ** 1.514, 0), axis=1)
def kova(T, P, lat, I, Mmax=0.76, Mmin=0.01, alph=0.093, m_th=4.886, mu_th=5.80, rootd=1000., M0=0.2):
    """Defter 1 ile aynı VS-Lite kova modeli; ısı indeksi I dışarıdan (taban iklim) verilir"""
    S_, N = T.shape; gu = gun_uzunlugu(lat)
    ngun = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]); L = (ngun / 30)[None, :] * (gu / 12)
    a = 6.75e-7 * I**3 - 7.71e-5 * I**2 + 1.792e-2 * I + 0.49
    M = np.empty((S_, N), np.float32); m = np.full(S_, M0)
    for t in range(N):
        Tt, Pt, Lt = T[:, t], P[:, t], L[:, t % 12]
        Ep = np.where(Tt <= 0, 0., np.where(Tt < 26.5, 16 * Lt * (10 * np.maximum(Tt, 0) / np.maximum(I, 1e-6)) ** a, -415.85 + 32.25 * Tt - 0.43 * Tt**2))
        Ep = np.maximum(Ep, 0); E = Ep * (m / Mmax)
        G_ = mu_th * alph / (1 + mu_th) * m * rootd; R_ = Pt * (m / Mmax) ** m_th + (alph / (1 + mu_th)) * m * rootd
        m = np.clip(m + (Pt - E - R_ - G_) / rootd, Mmin, Mmax); M[:, t] = m
    return M
es = lambda T: 0.6108 * np.exp(17.27 * T / (T + 237.3))
@torch.no_grad()
@torch.no_grad()
def tahmin_G(model, G, bs=65536):
    n = len(G['rp']); out = []
    for i in range(0, n, bs): out.append(model(None, None, G={k: v[i:i+bs] for k, v in G.items()})[0])
    return torch.cat(out)
def yukle(bolunme_ad, model_ad, tohum=0):
    kl = os.path.join(SONUC, f'{bolunme_ad.replace(":", "-")}__{model_ad}__s{tohum}')
    if not os.path.exists(os.path.join(kl, 'en_iyi.pt')): return None, None
    egt, _, test, _ = bolunme(bolunme_ad); kat_gor_ayarla(egt)
    m = yap(model_ad); m.load_state_dict(torch.load(os.path.join(kl, 'en_iyi.pt'), map_location=DEV)); m.eval()
    return m, test

import xgboost as xgb
print('xgboost', xgb.__version__)
NG_SEC = 2_000_000   # XGBoost eğitim alt örneklemi (halka); doğrulama 300 bin
def ozellik_ag(b, gizle):
    """[B, 126]: 24 ay × (T, logP, M, VPD) + log yarıçap + g_E ağırlıklı mevsim ortalamaları (önceki/bu yıl) + alan + cins, tür"""
    X, s = pencere(b); w = X[..., 4]
    mev = [((X[:, a:a+12, c] * w[:, a:a+12]).sum(1) / w[:, a:a+12].sum(1).clamp_min(1e-6))[:, None] for a in (0, 12) for c in range(4)]
    g = site_genus[s].float(); sp = site_species[s].float()
    if gizle: g = torch.full_like(g, float('nan')); sp = torch.full_like(sp, float('nan'))
    return torch.cat([X[..., :4].flatten(1), torch.log(R_rp[b] + 0.1)[:, None]] + mev + [DESC[s], ENV[s], g[:, None], sp[:, None]], 1)
def ozellik_np(idx, gizle, bs=500_000):
    return np.concatenate([ozellik_ag(idx[i:i+bs], gizle).cpu().numpy() for i in range(0, len(idx), bs)]).astype(np.float32)
FT = ['q'] * (96 + 1 + 8 + DESC.shape[1] + ENV.shape[1]) + ['c', 'c']
XDEV = 'cuda' if DEV == 'cuda' else 'cpu'
def kaydet(b, ad, idx, mu, sg, ek):
    kl = os.path.join(SONUC, f'{b.replace(":", "-")}__{ad}__s0'); os.makedirs(kl, exist_ok=True)
    np.savez_compressed(os.path.join(kl, 'test_tahmin.npz'), idx=idx.astype(np.int32), mu=mu.astype(np.float16), sigma=sg.astype(np.float16))
    json.dump(ek, open(os.path.join(kl, 'bitti.json'), 'w'))
def bitti(b, ad): return os.path.exists(os.path.join(SONUC, f'{b.replace(":", "-")}__{ad}__s0', 'bitti.json'))

def dkm_ozellik(b, gizle, g_gor):
    """Doğrusal karma model tasarımı: sabit, log r, (log r)², 8 mevsim iklimi, DESC, ENV, cins (gizlide görülen cinslerin ortalaması)"""
    X, s = pencere(b); w = X[..., 4]
    mev = [((X[:, a:a+12, c] * w[:, a:a+12]).sum(1) / w[:, a:a+12].sum(1).clamp_min(1e-6))[:, None] for a in (0, 12) for c in range(4)]
    lr = torch.log(R_rp[b] + 0.1)[:, None]
    oh = F.one_hot(site_genus[s], NG + 1).float()
    if gizle: oh = g_gor.float()[None, :].expand(len(b), -1) / g_gor.sum()
    return torch.cat([torch.ones_like(lr), lr, lr ** 2] + mev + [DESC[s], ENV[s], oh], 1), s

In [ ]:
# --- Ortam bilgisi: kullanılan sürümler (yeniden üretilebilirlik) ---
import sys, platform, importlib
def surum(p):
    try: return importlib.import_module(p).__version__
    except Exception: return 'yok'
SURUM = {p: surum(p) for p in ['numpy', 'pandas', 'scipy', 'torch', 'xgboost', 'xarray', 'zarr', 'fsspec', 'gcsfs']}
SURUM.update(python=platform.python_version(), cuda=(torch.version.cuda if torch.cuda.is_available() else 'yok'))
print(SURUM); json.dump(SURUM, open(os.path.join(ANALIZ, 'K_surumler.json'), 'w'), indent=1)

In [ ]:
# --- K1. Karşı-olgusal iklim ---
# ISIMIP3a GSWP3-W5E5: obsclim (gözlenen) ve counterclim (insan kaynaklı uzun dönemli ısınma eğilimi çıkarılmış; Mengel ve ark. 2021).
# Modeller CRU TS ile eğitildiği için karşı-olgusal iklim, iki ISIMIP senaryosu arasındaki FARK CRU iklimine uygulanarak kurulur:
#   T_cf = T_CRU − (T_obs − T_cf,ISIMIP)      P_cf = P_CRU × (P_cf / P_obs)_ISIMIP (0,3–3 arasında; P_obs < 1 mm ise 1)
#   VPD_cf = max(0, VPD_CRU − (VPD_obs − VPD_cf)_ISIMIP)      toprak nemi: aynı VS-Lite kova modeliyle yeniden hesaplanır
# Böylece olgusal girdi, modellerin eğitildiği iklimin kendisidir; karşı-olgusal girdi yalnızca iklim değişikliği sinyali kadar farklıdır.
# Alan tanımlayıcıları (1901–1960 normalleri) ve ağaç boyutları iki senaryoda aynıdır: fark yalnızca yıllık iklimden gelir.
KL_IS = os.path.join(ANA, 'IKLIM_ISIMIP')
YIL_SON = 2019                                   # ISIMIP3a obsclim/counterclim 1901–2019
kod2s = {c: i for i, c in enumerate(soz['site_code'])}
def isimip_oku(sen):
    df = pd.read_csv(os.path.join(KL_IS, f'isimip_{sen}_aylik.csv.gz'), usecols=['studyCode', 'year', 'month', 'tmp', 'pre', 'vpd'])
    df = df[df.studyCode.isin(list(kod2s))]
    s_ = df.studyCode.map(kod2s).to_numpy(); y_ = df.year.to_numpy() - YIL0; m_ = df.month.to_numpy() - 1
    out = {}
    for v in ['tmp', 'pre', 'vpd']:
        a = np.full((S, NY, 12), np.nan, np.float32); a[s_, y_, m_] = df[v].to_numpy(np.float32); out[v] = a
    return out
OBS = isimip_oku('obsclim'); CFI = isimip_oku('counterclim')
isimip_var = ~np.isnan(OBS['tmp']).all(axis=(1, 2))
print('ISIMIP verisi olan alan:', int(isimip_var.sum()), '/', S)
dT = np.nan_to_num(OBS['tmp'] - CFI['tmp'])                                   # [S, NY, 12]; 2020–2024 ve verisi olmayan alanlarda 0
oranP = np.where(OBS['pre'] > 1.0, CFI['pre'] / np.maximum(OBS['pre'], 1e-6), 1.0)
oranP = np.clip(np.nan_to_num(oranP, nan=1.0), 0.3, 3.0)
dV = np.nan_to_num(OBS['vpd'] - CFI['vpd'])
del OBS, CFI; gc.collect()
LATs = d['site_lat'].astype(float)
T_f = CL[..., 0].astype(np.float64); P_f = CL[..., 1].astype(np.float64); V_f = CL[..., 3].astype(np.float64)
T_c = T_f - dT; P_c = P_f * oranP; V_c = np.maximum(V_f - dV, 0)
I_tam = isi_indeksi(T_f.reshape(S, -1))                                       # Defter 1 ile aynı ısı indeksi (iki senaryoda sabit)
M_f = kova(T_f.reshape(S, -1), P_f.reshape(S, -1), LATs, I_tam)
M_c = kova(T_c.reshape(S, -1), P_c.reshape(S, -1), LATs, I_tam)
print('Kontrol — yeniden hesaplanan olgusal toprak nemi ile Defter 1 farkı (en büyük mutlak):',
      round(float(np.nanmax(np.abs(M_f - d['soil'].reshape(S, -1)))), 5))
ozel_c = np.stack([T_c.reshape(S, -1), np.log1p(np.maximum(P_c.reshape(S, -1), 0)), M_c, V_c.reshape(S, -1),
                   np.tile(d['gE'], (1, NY))], -1).astype(np.float32)
OZ_F = OZ; OZ_C = t(ozel_c); del ozel_c; gc.collect()
# İklim değişikliği sinyalinin alan düzeyi özeti (1990–2019)
j0, j1 = 1990 - YIL0, YIL_SON - YIL0 + 1
Mf3, Mc3 = M_f.reshape(S, NY, 12), M_c.reshape(S, NY, 12)
SINYAL = pd.DataFrame(dict(studyCode=soz['site_code'], isimip_var=isimip_var,
                           dT_C=dT[:, j0:j1].mean((1, 2)),
                           P_oran=P_f[:, j0:j1].sum((1, 2)) / np.maximum(P_c[:, j0:j1].sum((1, 2)), 1e-6),
                           dVPD_kPa=dV[:, j0:j1].mean((1, 2)), dM=(Mf3 - Mc3)[:, j0:j1].mean((1, 2))))
SINYAL.to_csv(os.path.join(ANALIZ, 'K0_iklim_sinyali_1990_2019.csv'), index=False)
print(SINYAL[SINYAL.isimip_var].describe().round(3).to_string())

In [ ]:
# --- M0. Ortak araçlar ---
R_w_n = R_w.cpu().numpy(); R_ser_n = R_ser.cpu().numpy()
NYa = YIL_SON - YIL0 + 1; YILLAR_A = np.arange(YIL0, YIL_SON + 1)
gen_ad = np.array(soz['genus'] + ['Bilinmeyen'])[np.minimum(site_genus.cpu().numpy(), len(soz['genus']))]
soguk = desc[:, 2] <= np.quantile(desc[:, 2], 0.25); kurak = desc[:, 5] <= np.quantile(desc[:, 5], 0.25)
GRUP = {'Tüm alanlar': np.ones(S, bool), 'En soğuk %25': soguk, 'En kurak %25': kurak,
        **{f'Bölge: {b}': site_bolge == b for b in np.unique(site_bolge) if b != 'Diger'}}
CINS = [g for g in pd.Series(gen_ad).value_counts().index if (gen_ad == g).sum() >= 40][:14]
rngM = np.random.default_rng(2031)
def kronoloji(y, mask):
    """Seri içi 31 yıllık hareketli ortalamadan sapma → alan × yıl ortalaması [S, NYa] (Defter 9 ve 12 ile aynı)"""
    ser = R_ser_n[mask]; yil = YIL_n[mask]; s = SITE_n[mask]; yv = y[mask].astype(np.float64)
    o = np.lexsort((yil, ser)); ser, yil, s, yv = ser[o], yil[o], s[o], yv[o]
    n_ = len(yv); pos = np.arange(n_)
    bas_ = np.r_[0, np.flatnonzero(np.diff(ser)) + 1]; uzun = np.diff(np.r_[bas_, n_])
    sb = np.repeat(bas_, uzun); se_ = sb + np.repeat(uzun, uzun)
    lo = np.maximum(sb, pos - 15); hi = np.minimum(se_, pos + 16)
    cs = np.r_[0, np.cumsum(yv)]; say = hi - lo
    iz = np.where(say >= 15, yv - (cs[hi] - cs[lo]) / np.maximum(say, 1), np.nan); k_ = np.isfinite(iz)
    del cs, lo, hi, sb, se_, pos
    key = s[k_].astype(np.int64) * NYa + (yil[k_].astype(np.int64) - YIL0)
    top = np.bincount(key, iz[k_], S * NYa); n = np.bincount(key, None, S * NYa)
    return np.where(n > 0, top / np.maximum(n, 1), np.nan).reshape(S, NYa)
def hareketli_sapma(A):
    n = A.shape[-1]; cs = np.concatenate([np.zeros(A.shape[:-1] + (1,)), np.cumsum(A, -1)], -1)
    i = np.arange(n); lo = np.maximum(i - 15, 0); hi = np.minimum(i + 16, n)
    return A - (cs[..., hi] - cs[..., lo]) / (hi - lo)
def alan_r(A, B, j0, j1, nmin=30):
    """Alan bazında Pearson r (yıllar j0..j1)"""
    a, b = A[:, j0:j1], B[:, j0:j1]; ok = np.isfinite(a) & np.isfinite(b); n = ok.sum(1)
    a = np.where(ok, a, 0); b = np.where(ok, b, 0)
    ma = a.sum(1) / np.maximum(n, 1); mb = b.sum(1) / np.maximum(n, 1)
    ca = np.where(ok, a - ma[:, None], 0); cb = np.where(ok, b - mb[:, None], 0)
    r = (ca * cb).sum(1) / np.sqrt((ca ** 2).sum(1) * (cb ** 2).sum(1) + 1e-12)
    return np.where(n >= nmin, r, np.nan)
def boot_med(x, B=1000):
    x = x[np.isfinite(x)]
    if len(x) < 20: return np.nan, np.nan, np.nan
    bs = [np.median(rngM.choice(x, len(x))) for _ in range(B)]
    return float(np.median(x)), float(np.quantile(bs, 0.025)), float(np.quantile(bs, 0.975))
# Aylık iklim [S, NYa, 12]: olgusal (CRU) ve karşı-olgusal (K1)
Tf3 = T_f[:, :NYa]; Tc3 = T_c[:, :NYa]; Mf3 = M_f.reshape(S, NY, 12)[:, :NYa]; Mc3 = M_c.reshape(S, NY, 12)[:, :NYa]
GE3 = d['gE'][:, None, :]
def gs(A): return (A * GE3).sum(-1) / GE3.sum(-1)            # büyüme mevsimi (g_E ağırlıklı) yıllık değer
soguk_ay = np.argsort(T_f[:, :60].mean(1), axis=1)[:, :3]     # alanın en soğuk 3 ayı (1901–1960 iklimi)
def kis(A): return np.take_along_axis(A, np.broadcast_to(soguk_ay[:, None, :], A.shape[:2] + (3,)), 2).mean(-1)
IKL_F = dict(kurak=gs(Mf3), sicak=gs(Tf3), soguk=kis(Tf3)); IKL_C = dict(kurak=gs(Mc3), sicak=gs(Tc3), soguk=kis(Tc3))
OLAY_AD = {'kurak': 'Kurak yıl (büyüme mevsimi toprak nemi ≤ %10)', 'sicak': 'Sıcak yıl (büyüme mevsimi sıcaklığı ≥ %90)', 'soguk': 'Sert kış (en soğuk 3 ay ≤ %10)'}
YON = dict(kurak=-1, sicak=1, soguk=-1)

In [ ]:
# --- M1. SORU 1 — Kim daha iyi öğreniyor? Tüm yöntemler 1902–1960 ile kurulur, 1961–2019'da yıllar arası büyüme değişimiyle sınanır ---
# Ölçüt: alan bazında gözlenen kronoloji ile tahmin edilen kronoloji arasındaki korelasyon (1961–2019; aynı halkalar, aynı detrend).
_, _, test_z, _ = bolunme('zaman')
mask_t = np.zeros(NR, bool); mask_t[test_z.cpu().numpy()] = True; mask_t &= (R_w_n > 0) & (YIL_n <= YIL_SON)
IZ_T = kronoloji(np.log(np.maximum(R_w_n, 1e-6)), mask_t)                     # gözlenen, test halkaları
mask_c = (R_w_n > 0) & (YIL_n >= YIL0 + 1) & (YIL_n <= 1960)
IZ_C = kronoloji(np.log(np.maximum(R_w_n, 1e-6)), mask_c)                     # gözlenen, kalibrasyon dönemi (≤1960)
J60 = 1960 - YIL0 + 1
TAHMIN = {}
AD_Z = {'teori_ortam': 'T+S (teori + YZ)', 'teori_vpd_ortam': 'T_D+S (teori + YZ)', 'serbest_ortam': 'U+S (YZ, teorisiz)',
        'xgb_ayar': 'XGBoost (YZ, teorisiz)', 'dkm': 'Karma model (klasik)'}
for m, ad in AD_Z.items():
    r_ = topluluk('zaman', m)
    if r_ is None and m == 'xgb_ayar' and os.path.exists(os.path.join(SONUC, 'zaman__xgb_model.json')):   # Defter 12'de kaydedilen model
        K_ = xgb.Booster(); K_.load_model(os.path.join(SONUC, 'zaman__xgb_model.json')); tur = int(json.load(open(os.path.join(SONUC, 'zaman__xgb_model.json.tur')))['tur'])
        kat_gor_ayarla(bolunme('zaman')[0]); ix = test_z
        mu = np.concatenate([K_.predict(xgb.DMatrix(ozellik_np(ix[i:i + 1_000_000], False), feature_types=FT, enable_categorical=True), iteration_range=(0, tur + 1))
                             for i in range(0, len(ix), 1_000_000)]); r_ = (ix.cpu().numpy(), mu)
    if r_ is None: print('Zamansal tahmin yok:', ad); continue
    a = np.full(NR, np.nan, np.float32); a[r_[0]] = r_[1]
    TAHMIN[ad] = kronoloji(a, mask_t & np.isfinite(a))
# Klasik: tepki fonksiyonu (temel bileşen regresyonu) ve VS-Lite, yalnız 1902–1960 ile kalibre
def pencere16(A):
    out = np.full(A.shape[:2] + (16,), np.nan, np.float32); out[:, 1:, :7] = A[:, :-1, 5:12]; out[:, :, 7:] = A[:, :, :9]; return out
Xf = np.concatenate([pencere16(Tf3.astype(np.float32)), pencere16(np.log1p(np.maximum(P_f[:, :NYa], 0)).astype(np.float32))], 2)
Xa = np.full_like(Xf, np.nan); Xa[:, 1:] = np.moveaxis(hareketli_sapma(np.moveaxis(Xf[:, 1:], 1, 2).astype(np.float64)), 2, 1)
RF_T = np.full((S, NYa), np.nan)
for s in range(S):
    ok = np.isfinite(IZ_C[s, :J60]) & np.isfinite(Xa[s, :J60]).all(1)
    if ok.sum() < 30: continue
    Z = Xa[s, :J60][ok]; y = IZ_C[s, :J60][ok] - IZ_C[s, :J60][ok].mean(); sd = Z.std(0) + 1e-9; mu_ = Z.mean(0); Zs = (Z - mu_) / sd
    U_, Sv, Vt = np.linalg.svd(Zs, full_matrices=False); k_ = (Sv ** 2 / (ok.sum() - 1)) > 1
    if not k_.any(): k_[0] = True
    bp = np.linalg.lstsq(Zs @ Vt[k_].T, y, rcond=None)[0]; b = Vt[k_].T @ bp
    RF_T[s] = ((Xa[s] - mu_) / sd) @ b
TAHMIN['Tepki fonksiyonu (klasik)'] = RF_T
T1g, T2g = [0., 1.5, 3., 4.5, 6., 7.5, 9.], [10., 12., 14., 16., 18., 20., 22., 24.]
M1g, M2g = [0.01, 0.02, 0.03], [0.1, 0.2, 0.3, 0.4, 0.5]
PT = t([(a, b) for a in T1g for b in T2g], torch.float32); PM = t([(a, b) for a in M1g for b in M2g], torch.float32)
TFt = t(Tf3, torch.float32); MFt = t(Mf3, torch.float32); GEt = t(d['gE'], torch.float32)
IZc = IZ_C.copy(); IZc[:, J60:] = np.nan; IZt = t(np.nan_to_num(IZc), torch.float32); IZm = t(np.isfinite(IZc), torch.float32)
def rampa(X, P): return ((X[:, None] - P[None, :, 0, None, None]) / (P[None, :, 1, None, None] - P[None, :, 0, None, None])).clamp(0, 1)
VS_T = np.full((S, NYa), np.nan)
with torch.no_grad():
    for i in range(0, S, 32):
        sl = slice(i, min(i + 32, S)); m_ = IZm[sl]; n_ = m_.sum(1)
        W = (torch.minimum(rampa(TFt[sl], PT)[:, :, None], rampa(MFt[sl], PM)[:, None]) * GEt[sl, None, None, None, :]).sum(-1).flatten(1, 2)
        mw = (W * m_[:, None]).sum(-1) / n_.clamp_min(1)[:, None]; Wc_ = (W - mw[..., None]) * m_[:, None]
        yi = (IZt[sl] - (IZt[sl] * m_).sum(1, keepdim=True) / n_.clamp_min(1)[:, None]) * m_
        cov = (Wc_ * yi[:, None]).sum(-1); vw = (Wc_ ** 2).sum(-1); vy = (yi ** 2).sum(-1)
        r = torch.where(vw > 1e-10, cov / torch.sqrt(vw * vy[:, None]).clamp_min(1e-12), torch.full_like(cov, -2))
        en = r.argmax(1); eg = cov.gather(1, en[:, None])[:, 0] / vw.gather(1, en[:, None])[:, 0].clamp_min(1e-12)
        Wb = W.gather(1, en[:, None, None].expand(-1, 1, W.shape[-1]))[:, 0]
        pred = ((Wb - mw.gather(1, en[:, None])) * eg[:, None]).cpu().numpy()
        VS_T[sl] = np.where((n_ >= 30).cpu().numpy()[:, None], pred, np.nan)
VS_T = hareketli_sapma(np.nan_to_num(VS_T)) + np.where(np.isfinite(VS_T), 0, np.nan)     # tahmin serisi de aynı şekilde detrend
TAHMIN['VS-Lite (klasik)'] = VS_T
j61 = 1961 - YIL0
M1, M1a = [], pd.DataFrame(dict(studyCode=soz['site_code'], cins=gen_ad, bolge=site_bolge))
for ad, A in TAHMIN.items():
    r = alan_r(IZ_T, A, j61, NYa); M1a[ad] = r
    for g_ad, msk in {**GRUP, **{f'Cins: {c}': gen_ad == c for c in CINS}}.items():
        med, lo, hi = boot_med(r[msk])
        if np.isfinite(med): M1.append(dict(yontem=ad, grup=g_ad, n_alan=int(np.isfinite(r[msk]).sum()), medyan_r=med, alt95=lo, ust95=hi,
                                             r_0_3_ustu_oran=float(np.nanmean(r[msk] > 0.3))))
M1 = pd.DataFrame(M1); M1.to_csv(os.path.join(ANALIZ, 'M1_beceri_ozet.csv'), index=False); M1a.to_csv(os.path.join(ANALIZ, 'M1a_beceri_alan.csv'), index=False)
print('SORU 1 — 1961–2019 yıllar arası büyüme değişimini yakalama (alan bazında medyan r; yöntemler yalnız 1960\'a kadarki veriyi gördü):')
print(M1[M1.grup == 'Tüm alanlar'][['yontem', 'n_alan', 'medyan_r', 'alt95', 'ust95', 'r_0_3_ustu_oran']].round(3).to_string(index=False))
# Eşleştirilmiş karşılaştırma: T+S ile her yöntem (aynı alanlar), alan bootstrap
ref = 'T+S (teori + YZ)'; KAR = []
if ref in M1a:
    for ad in TAHMIN:
        if ad == ref: continue
        ok = np.isfinite(M1a[ref]) & np.isfinite(M1a[ad]); dlt = (M1a[ref] - M1a[ad])[ok].values
        bs = [rngM.choice(dlt, len(dlt)).mean() for _ in range(1000)]
        KAR.append(dict(karsilastirma=f'{ref} − {ad}', n_alan=int(ok.sum()), ort_r_farki=float(dlt.mean()), alt95=float(np.quantile(bs, 0.025)), ust95=float(np.quantile(bs, 0.975)),
                        T_S_daha_iyi_alan_orani=float((dlt > 0).mean())))
    KAR = pd.DataFrame(KAR); KAR.to_csv(os.path.join(ANALIZ, 'M1b_beceri_karsilastirma.csv'), index=False); print(KAR.round(3).to_string(index=False))
print(M1[M1.grup != 'Tüm alanlar'].pivot(index='grup', columns='yontem', values='medyan_r').round(2).to_string())

In [ ]:
# --- N0. Ortak araçlar (Defter 14) ---
def kronoloji_w(y, mask, yari=15, nmin=None):
    """kronoloji() ile aynı; hareketli ortalama penceresi 2·yari+1 yıl (duyarlılık analizi için)"""
    nmin = yari if nmin is None else nmin
    ser = R_ser_n[mask]; yil = YIL_n[mask]; s = SITE_n[mask]; yv = y[mask].astype(np.float64)
    o = np.lexsort((yil, ser)); ser, yil, s, yv = ser[o], yil[o], s[o], yv[o]
    n_ = len(yv); pos = np.arange(n_)
    bas_ = np.r_[0, np.flatnonzero(np.diff(ser)) + 1]; uzun = np.diff(np.r_[bas_, n_])
    sb = np.repeat(bas_, uzun); se_ = sb + np.repeat(uzun, uzun)
    lo = np.maximum(sb, pos - yari); hi = np.minimum(se_, pos + yari + 1)
    cs = np.r_[0, np.cumsum(yv)]; say = hi - lo
    iz = np.where(say >= nmin, yv - (cs[hi] - cs[lo]) / np.maximum(say, 1), np.nan); k_ = np.isfinite(iz)
    del cs, lo, hi, sb, se_, pos
    key = s[k_].astype(np.int64) * NYa + (yil[k_].astype(np.int64) - YIL0)
    top = np.bincount(key, iz[k_], S * NYa); n = np.bincount(key, None, S * NYa)
    return np.where(n > 0, top / np.maximum(n, 1), np.nan).reshape(S, NYa)
mask_all = (R_w_n > 0) & (YIL_n >= YIL0 + 1) & (YIL_n <= YIL_SON)
LOGW = np.log(np.maximum(R_w_n, 1e-6))
IZ_A = kronoloji_w(LOGW, mask_all, 15)
yuzde = lambda x: 100 * (np.exp(x) - 1)
GRUPLAR = {**GRUP, **{f'Cins: {c}': gen_ad == c for c in CINS}}
j90, j19 = 1990 - YIL0, YIL_SON - YIL0 + 1; J60 = 1960 - YIL0 + 1
print('Kronoloji hazır:', int(np.isfinite(IZ_A).any(1).sum()), 'alan')

In [ ]:
# --- N1. Uç olay kanalı ---
OLAY_K = list(OLAY_AD)
def olay_matris(q=0.10, j0=1, j1=J60):
    """{olay: (olgusal [S,NYa] bool, karşı-olgusal [S,NYa] bool)}; eşik: olgusal iklimin j0..j1 yıllarındaki q / 1−q nicelikleri"""
    D = {}
    for k in OLAY_K:
        qq = 1 - q if YON[k] > 0 else q
        e = np.nanquantile(IKL_F[k][:, j0:j1], qq, axis=1)[:, None]
        D[k] = (IKL_F[k] >= e, IKL_C[k] >= e) if YON[k] > 0 else (IKL_F[k] <= e, IKL_C[k] <= e)
    return D
def toplu_ekk(X, y, m, ridge, nmin=30):
    """Alan başına EKK (vektörel). X [S,T,p], y [S,T], m [S,T] → katsayı [S,p]"""
    Xm = X * m[..., None]; ym = np.where(m, y, 0.)
    XtX = np.einsum('stp,stq->spq', Xm, Xm) + np.diag(ridge)[None]; Xty = np.einsum('stp,st->sp', Xm, ym)
    b = np.linalg.solve(XtX, Xty[..., None])[..., 0]
    b[m.sum(1) < nmin] = np.nan
    return b
def tasarim(D, gecikme=True):
    kol = [D[k][0].astype(np.float64) for k in OLAY_K]
    if gecikme: kol += [np.concatenate([np.zeros((S, 1)), D[k][0][:, :-1]], 1).astype(np.float64) for k in OLAY_K]
    return np.stack([np.ones((S, NYa))] + kol, -1)
def olay_etkisi(IZ, D, gecikme=True, j0=2, j1=NYa, en_az_olay=3):
    """Alan × olay toplam (olay yılı + ertesi yıl) log etkisi [S, 3]"""
    X = tasarim(D, gecikme)[:, j0:j1]; y = IZ[:, j0:j1]; m = np.isfinite(y)
    p = X.shape[-1]; ridge = np.r_[1e-6, np.ones(p - 1)]          # olay göstergelerinde hafif büzme (az olaylı alanlar)
    b = toplu_ekk(X, y, m, ridge)
    say = (X[..., 1:4] * m[..., None]).sum(1)                        # alan başına olay sayısı
    top = b[:, 1:4] + (b[:, 4:7] if gecikme else 0)
    return np.where(say >= en_az_olay, top, 0.0), b, say
def siklik_farki(D, y0=1990, y1=YIL_SON):
    a, b = y0 - YIL0, y1 - YIL0 + 1
    Ff = np.stack([D[k][0][:, a:b].mean(1) for k in OLAY_K], 1); Fc = np.stack([D[k][1][:, a:b].mean(1) for k in OLAY_K], 1)
    return Ff, Fc
def kanal(IZ, D, gecikme=True, j0=2, j1=NYa):
    E, b, say = olay_etkisi(IZ, D, gecikme, j0, j1)
    Ff, Fc = siklik_farki(D)
    Lk = E * (Ff - Fc)
    ok = isimip_var & np.isfinite(Lk).all(1) & np.isfinite(IZ[:, j90:j19]).any(1)
    return dict(E=E, Ff=Ff, Fc=Fc, Lk=Lk, L=Lk.sum(1), ok=ok, say=say)

In [ ]:
# --- Sınırlayıcı faktör fonksiyonu (Defter 14 ile aynı) ---
u_, ilk = np.unique(ser_site.cpu().numpy(), return_index=True); ser0 = np.zeros(S, np.int64); ser0[u_] = ilk; SER0 = t(ser0, torch.long); YRS = np.arange(1990, YIL_SON + 1)
@torch.no_grad()
def sinir(model, OZx, vpd, bs=65536):
    """[S, yıl, k+1]: k faktörün büyüme ağırlıklı sınırlama payı ve iklim katsayısı C"""
    s_all = torch.arange(S, device=DEV).repeat_interleave(len(YRS)); y_all = t(np.tile(YRS, S), torch.long); out = []
    for i in range(0, len(s_all), bs):
        s = s_all[i:i + bs]; y = y_all[i:i + bs]
        X = OZx[s[:, None], ((y - 1 - YIL0) * 12)[:, None] + torch.arange(24, device=DEV)[None, :]]
        G = alan_girdi(s); p = model.parametreler(G)
        Fs = [monoton(X[..., 0], p['aT'], p['cT'], p['sT']), monoton(X[..., 2], p['aM'], p['cM'], p['sM'])]
        if vpd: Fs.append(1 - monoton(X[..., 3], p['aD'], p['cD'], p['sD']))
        Fs = torch.stack(Fs, -1); w = p['omega'] * X[..., 4]
        wn = w / w.sum(1, keepdim=True).clamp_min(1e-9); arg = Fs.argmin(-1)
        pay = torch.stack([(wn * (arg == j)).sum(1) for j in range(Fs.shape[-1])], 1)
        C = (w * Fs.min(-1).values).sum(1).clamp_min(1e-6)
        out.append(torch.cat([pay, C[:, None]], 1))
    return torch.cat(out).reshape(S, len(YRS), -1).cpu().numpy()

In [ ]:
# --- Q0. Ortak araçlar: blok boyutu parametreli bootstrap ---
D10 = olay_matris(0.10); KN = kanal(IZ_A, D10)
LATn, LONn = d['site_lat'].astype(float), d['site_lon'].astype(float)
def bloklar(deg): return (np.floor((LATn + 90) / deg) * 1000 + np.floor((LONn + 180) / deg)).astype(int)
BLOK10 = bloklar(10)
rngQ = np.random.default_rng(2061); NB_ = 200 if AYAR['hizli_deneme'] else 1000
def blok_gruplar(idx, BL):
    b = BL[idx]; ub, inv = np.unique(b, return_inverse=True); return [idx[inv == i] for i in range(len(ub))]
def blok_ornek(gr): return np.concatenate([gr[i] for i in rngQ.integers(0, len(gr), len(gr))])
def blok_ci(f, idx, BL=BLOK10, B=NB_):
    gr = blok_gruplar(idx, BL); out = [f(blok_ornek(gr)) for _ in range(B)]; return np.quantile(out, [0.025, 0.975]), np.array(out)
AD11 = {'teori_ortam': 'T+S', 'teori_vpd_ortam': 'T_D+S', 'serbest_ortam': 'U+S', 'xgb': 'XGBoost', 'dkm': 'Karma model'}
u_, ilk = np.unique(ser_site.cpu().numpy(), return_index=True); ser0 = np.zeros(S, np.int64); ser0[u_] = ilk; SER0 = t(ser0, torch.long)
YRS = np.arange(1990, YIL_SON + 1)
pearson = lambda a, b: float(np.corrcoef(a, b)[0, 1])
print('Blok sayısı (10°):', len(np.unique(BLOK10)))

## Q1 — Modelleri AYNI hedef üzerinden karşılaştırma
Hakem itirazı: model atıfları (kademeli ısınma dahil) ile uç yıl tahmini (yalnız uç yıllar) aynı nicelik değil.
Çözüm: her modelin **bağımsız dönemde (1961–2019) tahmin ettiği büyüme serisine** gözlenen seriye uygulanan **aynı uç olay analizi** (Denklem 4–5) uygulanır.
Böylece her model için "uç yıllar yoluyla büyüme değişimi" L tahmini elde edilir ve gözlenen L ile aynı hedefte karşılaştırılır:
* alanlar arası korelasyon ve ortalama mutlak hata, sıcaklık / nem sınırlı sahalarda ortalama L (gözlenen ve tahmin);
* T+S ile her model arasındaki korelasyon farkı için 10° blok bootstrap aralığı (doğrudan fark testi).
Modeller yalnız 1902–1960 ile eğitildi (M1 ile aynı tahminler); klasik yöntemler de dahil.

In [ ]:
# --- Q1. Aynı hedef: tahmin edilen serilerde uç olay kanalı ---
LOBS = kanal(IZ_T, D10, j0=j61)                       # gözlenen, 1961–2019 test halkaları
kat_gor_ayarla(bolunme('zaman')[0])
ZT = [y_ for y_ in (yukle('zaman', 'teori_ortam', s_)[0] for s_ in range(5)) if y_ is not None]
PAYZ = np.mean([sinir(mm, OZ_F, False)[..., 0].mean(1) for mm in ZT], 0) if ZT else np.full(S, np.nan)   # ≤1960 eğitimli modelin sıcaklık payı
del ZT; torch.cuda.empty_cache()
LPRED = {ad: kanal(A, D10, j0=j61) for ad, A in TAHMIN.items()}
ortak = LOBS['ok'] & np.isfinite(LOBS['L']) & np.isfinite(PAYZ)
for K in LPRED.values(): ortak &= K['ok'] & np.isfinite(K['L'])
v = np.where(ortak)[0]; Tm, Mm = v[PAYZ[v] > 0.5], v[PAYZ[v] <= 0.5]
ref = 'T+S (teori + YZ)'
Q1 = [dict(yontem='Gözlenen', n_alan=len(v), r_gozlenenle=1.0, ort_mutlak_hata=0.0, L_tum=float(yuzde(LOBS['L'][v].mean())),
           L_sicaklik_sinirli=float(yuzde(LOBS['L'][Tm].mean())), L_nem_sinirli=float(yuzde(LOBS['L'][Mm].mean())), r_sicak_etkisi=1.0, r_kurak_etkisi=1.0)]
for ad, K in LPRED.items():
    r_ = pearson(K['L'][v], LOBS['L'][v])
    q = dict(yontem=ad, n_alan=len(v), r_gozlenenle=r_, ort_mutlak_hata=float(100 * np.mean(np.abs(K['L'][v] - LOBS['L'][v]))),
             L_tum=float(yuzde(K['L'][v].mean())), L_sicaklik_sinirli=float(yuzde(K['L'][Tm].mean())), L_nem_sinirli=float(yuzde(K['L'][Mm].mean())),
             r_sicak_etkisi=pearson(K['E'][v, 1], LOBS['E'][v, 1]), r_kurak_etkisi=pearson(K['E'][v, 0], LOBS['E'][v, 0]))
    if ad != ref and ref in LPRED:
        Kr = LPRED[ref]
        ci, _ = blok_ci(lambda i: pearson(Kr['L'][i], LOBS['L'][i]) - pearson(K['L'][i], LOBS['L'][i]), v)
        q['r_farki_TS_eksi_bu'] = pearson(Kr['L'][v], LOBS['L'][v]) - r_; q['fark_alt95'], q['fark_ust95'] = float(ci[0]), float(ci[1])
    Q1.append(q)
Q1 = pd.DataFrame(Q1); Q1.to_csv(os.path.join(ANALIZ, 'Q1_ayni_hedef_karsilastirma.csv'), index=False)
pd.set_option('display.width', 260)
print('Q1 — Aynı hedef (uç yıllar yoluyla büyüme değişimi, 1961–2019 bağımsız dönem; sınıflar ≤1960 eğitimli T+S sıcaklık payından):')
print(Q1.round(3).to_string(index=False))

## Q2 — Mekânsal dışarıda bırakma ile aynı karşılaştırma (blok modelleri varsa)
Her saha, kendi bloğunu görmemiş modelin tahminiyle temsil edilir (`blok-k` test tahminleri). Gözlenen ve tahmin edilen kronolojilerde aynı uç olay analizi (1903–2019).

In [ ]:
# --- Q2. Blok dışarıda bırakma tahminleri ---
Q2 = []; BLK = sorted({int(os.path.basename(p).split('__')[0].split('-')[1]) for p in glob.glob(os.path.join(SONUC, 'blok-*__*__s*'))})
print('Blok bölünmeleri:', BLK)
if BLK:
    BL_TAH = {}
    for m, ad in AD11.items():
        a = np.full(NR, np.nan, np.float32); var = False
        for k in BLK:
            r_ = topluluk(f'blok-{k}', m)
            if r_ is not None: a[r_[0]] = r_[1]; var = True
        if var: BL_TAH[ad] = a
    print('Blok tahmini olan modeller:', list(BL_TAH))
    mask_b = mask_all & np.isfinite(np.full(NR, 0.0))
    LOBS_B = KN
    for ad, a in BL_TAH.items():
        K = kanal(kronoloji_w(a, mask_all & np.isfinite(a), 15), D10)
        w_ = np.where(K['ok'] & LOBS_B['ok'] & np.isfinite(K['L']))[0]
        Q2.append(dict(yontem=ad, n_alan=len(w_), r_gozlenenle=pearson(K['L'][w_], LOBS_B['L'][w_]), ort_mutlak_hata=float(100 * np.mean(np.abs(K['L'][w_] - LOBS_B['L'][w_]))),
                       L_tum=float(yuzde(K['L'][w_].mean())), L_gozlenen=float(yuzde(LOBS_B['L'][w_].mean()))))
Q2 = pd.DataFrame(Q2); Q2.to_csv(os.path.join(ANALIZ, 'Q2_blok_ayni_hedef.csv'), index=False)
print('Q2 — Mekânsal dışarıda bırakma ile aynı hedef:'); print(Q2.round(3).to_string(index=False) if len(Q2) else '(blok tahmini yok)')

## Q3 — Tespit testinde modeller arası doğrudan fark ve blok boyutu duyarlılığı
* β(T+S) − β(diğer) için eşleştirilmiş blok bootstrap (aynı bloklar her iki modelde).
* Ana sonuçların 5°, 10° ve 20° bloklarla aralıkları.
* Blok ortak permütasyonu: aynı bloktaki tüm sahalar aynı rastgele zaman kaydırmasını alır (ortak yıllık değişkenlik korunur).

In [ ]:
# --- Q3. Tespit farkları, blok boyutu, blok permütasyonu ---
egt_z, _, test_z, _ = bolunme('zaman')
HZ = test_z[(R_yil[test_z] <= YIL_SON) & (R_w[test_z] > 0)]
if AYAR['hizli_deneme']: HZ = HZ[torch.randperm(len(HZ), device=DEV)[:300_000]].sort().values
ANZ = ser_site[R_ser[HZ]] * NYa + (R_yil[HZ] - YIL0); SAYZ = torch.bincount(ANZ, minlength=S * NYa).float()
def ay_z(v_): return (torch.bincount(ANZ, weights=v_.float(), minlength=S * NYa) / SAYZ.clamp_min(1)).reshape(S, NYa).cpu().numpy()
YOBS = ay_z(hedef(HZ)); VARZ = SAYZ.reshape(S, NYa).cpu().numpy() > 0
def donem(F_, VAR_, y0, y1, nmin=10):
    a, b = y0 - YIL0, y1 - YIL0 + 1; vv = VAR_[:, a:b] & np.isfinite(F_[:, a:b]); n = vv.sum(1)
    return np.where(n >= nmin, np.where(vv, F_[:, a:b], 0).sum(1) / np.maximum(n, 1), np.nan)
fark_d = lambda A: donem(A, VARZ, 1991, 2019) - donem(A, VARZ, 1961, 1990)
dGOZ = fark_d(YOBS); KLT = os.path.join(SONUC, 'tespit' + ('_deneme' if AYAR['hizli_deneme'] else ''))
SR = {}
for m in AD11:
    fl = sorted(glob.glob(os.path.join(KLT, f'{m}#*.npz'))) if m not in ('xgb', 'dkm') else [os.path.join(KLT, f'{m}.npz')]
    fl = [f for f in fl if os.path.exists(f)]
    if not fl: continue
    Zs = [np.load(f) for f in fl]; dF = fark_d(np.mean([z['f'] for z in Zs], 0)); dC = fark_d(np.mean([z['c'] for z in Zs], 0))
    SR[AD11[m]] = (dF - dC, dGOZ - dC)
ok = np.ones(S, bool)
for Sg, Rr in SR.values(): ok &= np.isfinite(Sg) & np.isfinite(Rr)
okv = np.where(ok)[0]
beta = lambda Sg, Rr, i: np.linalg.lstsq(np.c_[np.ones(len(i)), Sg[i]], Rr[i], rcond=None)[0][1]
gr = blok_gruplar(okv, BLOK10); BB = {ad: [] for ad in SR}
for _ in range(NB_):
    i_ = blok_ornek(gr)
    for ad, (Sg, Rr) in SR.items(): BB[ad].append(beta(Sg, Rr, i_))
Q3 = []
for ad in SR:
    b0 = beta(*SR[ad], okv); q = dict(model=ad, n_alan=len(okv), beta=float(b0), alt95=float(np.quantile(BB[ad], .025)), ust95=float(np.quantile(BB[ad], .975)))
    if ad != 'T+S' and 'T+S' in SR:
        dd = np.array(BB['T+S']) - np.array(BB[ad]); q['TS_eksi_bu'] = float(beta(*SR['T+S'], okv) - b0)
        q['fark_alt95'], q['fark_ust95'] = float(np.quantile(dd, .025)), float(np.quantile(dd, .975)); q['abs1_TS_eksi_abs1_bu'] = float(abs(beta(*SR['T+S'], okv) - 1) - abs(b0 - 1))
    Q3.append(q)
Q3 = pd.DataFrame(Q3); Q3.to_csv(os.path.join(ANALIZ, 'Q3_tespit_farklari.csv'), index=False)
print('Q3a — Tespit testi, ortak sahalar ve eşleştirilmiş blok bootstrap:'); print(Q3.round(3).to_string(index=False))
# Blok boyutu duyarlılığı
Q3b = []
for deg in (5, 10, 20):
    BLd = bloklar(deg)
    for g_ad in ['Tüm alanlar', 'En soğuk %25', 'En kurak %25']:
        w_ = np.where(KN['ok'] & GRUP[g_ad])[0]
        ci, _ = blok_ci(lambda i: KN['L'][i].mean(), w_, BLd)
        Q3b.append(dict(blok_derece=deg, n_blok=len(np.unique(BLd[w_])), grup=g_ad, ort_yuzde=float(yuzde(KN['L'][w_].mean())), alt95=float(yuzde(ci[0])), ust95=float(yuzde(ci[1]))))
    ci, _ = blok_ci(lambda i: pearson(PAYZ[i], LOBS['L'][i]), v, BLd)
    Q3b.append(dict(blok_derece=deg, n_blok=len(np.unique(BLd[v])), grup='r(≤1960 sıcaklık payı, 1961–2019 L)', ort_yuzde=pearson(PAYZ[v], LOBS['L'][v]), alt95=float(ci[0]), ust95=float(ci[1])))
Q3b = pd.DataFrame(Q3b); Q3b.to_csv(os.path.join(ANALIZ, 'Q3b_blok_boyutu.csv'), index=False)
print('Q3b — Blok boyutu duyarlılığı:'); print(Q3b.round(3).to_string(index=False))
# Blok ortak permütasyonu (ana grup ortalamaları için)
NPERM = 60 if AYAR['hizli_deneme'] else 300
def kaydir_blok(IZ, BL):
    ub, inv = np.unique(BL, return_inverse=True); k = rngQ.integers(10, NYa - 10, len(ub))[inv]
    idx = (np.arange(NYa)[None, :] - k[:, None]) % NYa; return np.take_along_axis(IZ, idx, 1)
PL = np.array([kanal(kaydir_blok(IZ_A, BLOK10), D10)['L'] for _ in range(NPERM)])
Q3c = []
for g_ad in ['Tüm alanlar', 'En soğuk %25', 'En kurak %25']:
    w_ = KN['ok'] & GRUP[g_ad]; obs = KN['L'][w_].mean(); nul = PL[:, w_].mean(1)
    Q3c.append(dict(grup=g_ad, gozlenen_yuzde=float(yuzde(obs)), p_blok_permutasyon=float((np.sum(np.abs(nul) >= abs(obs)) + 1) / (len(nul) + 1))))
Q3c = pd.DataFrame(Q3c); Q3c.to_csv(os.path.join(ANALIZ, 'Q3c_blok_permutasyon.csv'), index=False)
print('Q3c — Blok ortak permütasyonu:'); print(Q3c.round(4).to_string(index=False))

In [ ]:
# --- Q4. Saha sayısı farklarının kaynağı (Şekil 3b ile 3c) ---
atif_n = None
fl = sorted(glob.glob(os.path.join(SONUC, 'atif' + ('_deneme' if AYAR['hizli_deneme'] else ''), 'teori_ortam#*.npz')))
if fl:
    Fk = np.mean([np.load(f)['fark'] for f in fl], 0)[:, 1990 - YIL0:]; vv = np.isfinite(Fk) & (Fk != 0); atif_n = vv.sum(1) >= 10
Q4 = pd.DataFrame([dict(olcut='Uç olay kanalı geçerli (ISIMIP + 1990–2019 kronolojisi)', n=int(KN['ok'].sum())),
                   dict(olcut='Model atfı geçerli (1990–2019 içinde ≥10 halka yılı)', n=int(atif_n.sum()) if atif_n is not None else -1),
                   dict(olcut='İkisi birden', n=int((KN['ok'] & atif_n).sum()) if atif_n is not None else -1)])
Q4.to_csv(os.path.join(ANALIZ, 'Q4_saha_sayilari.csv'), index=False); print(Q4.to_string(index=False))
print('==== DEFTER 16 TAMAM ====  Tablolar:', ANALIZ, '(Q1–Q4)')